In [1]:
# Verificar que las librerías están instaladas correctamente
import pandas as pd
import geopandas as gpd
import folium
import matplotlib.pyplot as plt

print("pandas:", pd.__version__)
print("geopandas:", gpd.__version__)
print("folium:", folium.__version__)
print("✓ Todo listo para arrancar")

pandas: 3.0.2
geopandas: 1.1.3
folium: 0.20.0
✓ Todo listo para arrancar


In [2]:
# Mapa base de Lima
mapa = folium.Map(
    location=[-12.046374, -77.042793],  # Centro de Lima
    zoom_start=10,
    tiles='CartoDB positron'
)

# Guardar y verificar
mapa.save('mapa_base_lima.html')
print("✓ Mapa generado. Busca el archivo mapa_base_lima.html en la carpeta notebooks/")

✓ Mapa generado. Busca el archivo mapa_base_lima.html en la carpeta notebooks/


In [3]:
# Puntos de riesgo conocidos en Lima
# Fuente: CENEPRED y registros históricos de huaicos e inundaciones

zonas_riesgo = [
    {"nombre": "Chosica - Quebrada Quirio", "lat": -11.934, "lon": -76.697, "nivel": "Muy Alto", "tipo": "Huaico"},
    {"nombre": "Chosica - Quebrada Carossio", "lat": -11.940, "lon": -76.700, "nivel": "Muy Alto", "tipo": "Huaico"},
    {"nombre": "Carapongo", "lat": -11.987, "lon": -76.820, "nivel": "Alto", "tipo": "Inundación"},
    {"nombre": "Chaclacayo", "lat": -11.978, "lon": -76.768, "nivel": "Alto", "tipo": "Huaico"},
    {"nombre": "SJL - Quebrada Canto Grande", "lat": -11.957, "lon": -76.996, "nivel": "Alto", "tipo": "Huaico"},
    {"nombre": "Lurín - Zona baja", "lat": -12.273, "lon": -76.872, "nivel": "Medio", "tipo": "Inundación"},
    {"nombre": "Chillón - Puente Piedra", "lat": -11.865, "lon": -77.073, "nivel": "Alto", "tipo": "Inundación"},
    {"nombre": "Río Rímac - Ate", "lat": -12.026, "lon": -76.900, "nivel": "Medio", "tipo": "Inundación"},
]

# Colores por nivel de riesgo
colores = {
    "Muy Alto": "red",
    "Alto": "orange",
    "Medio": "yellow",
    "Bajo": "green"
}

# Iconos por tipo
iconos = {
    "Huaico": "warning-sign",
    "Inundación": "tint"
}

# Crear mapa
mapa = folium.Map(
    location=[-12.046374, -77.042793],
    zoom_start=11,
    tiles='CartoDB positron'
)

# Agregar cada zona de riesgo
for zona in zonas_riesgo:
    folium.Marker(
        location=[zona["lat"], zona["lon"]],
        popup=folium.Popup(
            f"""
            <b>{zona['nombre']}</b><br>
            Tipo: {zona['tipo']}<br>
            Nivel de riesgo: <b style='color:{colores[zona['nivel']]}'>{zona['nivel']}</b>
            """,
            max_width=200
        ),
        tooltip=zona["nombre"],
        icon=folium.Icon(
            color=colores[zona["nivel"]],
            icon=iconos[zona["tipo"]],
            prefix='glyphicon'
        )
    ).add_to(mapa)

# Guardar
mapa.save('mapa_riesgo_lima.html')
print(f"✓ Mapa generado con {len(zonas_riesgo)} zonas de riesgo")

✓ Mapa generado con 8 zonas de riesgo


C:\Users\ferge\AppData\Local\Temp\ipykernel_46584\2451468335.py:49: UserWarning: color argument of Icon should be one of: {'lightgreen', 'green', 'darkgreen', 'blue', 'lightblue', 'black', 'cadetblue', 'darkpurple', 'purple', 'lightred', 'lightgray', 'gray', 'darkred', 'beige', 'pink', 'white', 'orange', 'darkblue', 'red'}.
  icon=folium.Icon(


In [4]:
# Mapa profesional con ríos, leyenda y control de capas

mapa = folium.Map(
    location=[-12.046374, -77.042793],
    zoom_start=11,
    tiles='CartoDB positron'
)

# --- CAPA 1: Zonas de riesgo ---
capa_riesgo = folium.FeatureGroup(name="Zonas de Riesgo")

for zona in zonas_riesgo:
    folium.CircleMarker(
        location=[zona["lat"], zona["lon"]],
        radius=12,
        color=colores[zona["nivel"]],
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(
            f"""
            <b>{zona['nombre']}</b><br>
            Tipo: {zona['tipo']}<br>
            Nivel: <b>{zona['nivel']}</b>
            """,
            max_width=200
        ),
        tooltip=f"{zona['nombre']} — {zona['nivel']}"
    ).add_to(capa_riesgo)

capa_riesgo.add_to(mapa)

# --- CAPA 2: Ríos principales de Lima ---
capa_rios = folium.FeatureGroup(name="Ríos principales")

rios = [
    {
        "nombre": "Río Rímac",
        "color": "blue",
        "coords": [
            [-11.934, -76.697], [-11.978, -76.768],
            [-12.026, -76.900], [-12.046, -77.020],
            [-12.055, -77.120]
        ]
    },
    {
        "nombre": "Río Chillón",
        "color": "cadetblue",
        "coords": [
            [-11.865, -77.073], [-11.900, -77.050],
            [-11.930, -77.020], [-11.970, -77.000]
        ]
    },
    {
        "nombre": "Río Lurín",
        "color": "darkblue",
        "coords": [
            [-12.273, -76.872], [-12.220, -76.850],
            [-12.180, -76.830], [-12.150, -76.810]
        ]
    }
]

for rio in rios:
    folium.PolyLine(
        locations=rio["coords"],
        color=rio["color"],
        weight=3,
        opacity=0.8,
        tooltip=rio["nombre"]
    ).add_to(capa_rios)

capa_rios.add_to(mapa)

# --- LEYENDA ---
leyenda = """
<div style="
    position: fixed;
    bottom: 30px; left: 30px;
    background-color: white;
    border: 2px solid grey;
    border-radius: 8px;
    padding: 12px;
    font-size: 13px;
    z-index: 1000;
    box-shadow: 2px 2px 6px rgba(0,0,0,0.3);
">
    <b>Nivel de Riesgo</b><br>
    <span style="color:red">●</span> Muy Alto<br>
    <span style="color:orange">●</span> Alto<br>
    <span style="color:#cccc00">●</span> Medio<br>
    <hr style="margin:6px 0">
    <b>Ríos</b><br>
    <span style="color:blue">━</span> Rímac &nbsp;
    <span style="color:cadetblue">━</span> Chillón &nbsp;
    <span style="color:darkblue">━</span> Lurín
</div>
"""
mapa.get_root().html.add_child(folium.Element(leyenda))

# --- CONTROL DE CAPAS ---
folium.LayerControl().add_to(mapa)

# Guardar
mapa.save('mapa_riesgo_lima_v2.html')
print("✓ Mapa profesional generado: mapa_riesgo_lima_v2.html")

✓ Mapa profesional generado: mapa_riesgo_lima_v2.html


In [5]:
# Agregar proyecto de infraestructura y análisis de riesgo en zona de influencia

import pandas as pd
import numpy as np

# --- Proyecto simulado: Puente sobre el Río Rímac ---
proyecto = {
    "nombre": "Puente Río Rímac - Ate",
    "lat": -12.026,
    "lon": -76.900,
    "tipo": "Puente",
    "longitud_m": 120
}

# --- Calcular zonas de riesgo dentro de 5km del proyecto ---
def distancia_km(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

zonas_cercanas = []
for zona in zonas_riesgo:
    dist = distancia_km(proyecto["lat"], proyecto["lon"], zona["lat"], zona["lon"])
    if dist <= 5:
        zona_copia = zona.copy()
        zona_copia["distancia_km"] = round(dist, 2)
        zonas_cercanas.append(zona_copia)

zonas_cercanas.sort(key=lambda x: x["distancia_km"])

# --- Construir mapa final ---
mapa = folium.Map(
    location=[proyecto["lat"], proyecto["lon"]],
    zoom_start=12,
    tiles='CartoDB positron'
)

# Capas de riesgo
capa_riesgo = folium.FeatureGroup(name="Zonas de Riesgo")
for zona in zonas_riesgo:
    folium.CircleMarker(
        location=[zona["lat"], zona["lon"]],
        radius=12,
        color=colores[zona["nivel"]],
        fill=True,
        fill_opacity=0.7,
        tooltip=f"{zona['nombre']} — {zona['nivel']}",
        popup=folium.Popup(
            f"<b>{zona['nombre']}</b><br>Tipo: {zona['tipo']}<br>Nivel: <b>{zona['nivel']}</b>",
            max_width=200
        )
    ).add_to(capa_riesgo)
capa_riesgo.add_to(mapa)

# Ríos
capa_rios = folium.FeatureGroup(name="Ríos principales")
for rio in rios:
    folium.PolyLine(
        locations=rio["coords"],
        color=rio["color"],
        weight=3,
        opacity=0.8,
        tooltip=rio["nombre"]
    ).add_to(capa_rios)
capa_rios.add_to(mapa)

# Proyecto
capa_proyecto = folium.FeatureGroup(name="Proyecto de Infraestructura")
folium.Marker(
    location=[proyecto["lat"], proyecto["lon"]],
    popup=folium.Popup(
        f"""
        <b>🏗️ {proyecto['nombre']}</b><br>
        Tipo: {proyecto['tipo']}<br>
        Longitud: {proyecto['longitud_m']} m<br>
        <hr>
        <b>Zonas de riesgo en 5km: {len(zonas_cercanas)}</b>
        """,
        max_width=250
    ),
    tooltip=f"PROYECTO: {proyecto['nombre']}",
    icon=folium.Icon(color='blue', icon='home', prefix='glyphicon')
).add_to(capa_proyecto)

# Zona de influencia (5km)
folium.Circle(
    location=[proyecto["lat"], proyecto["lon"]],
    radius=5000,
    color='blue',
    fill=True,
    fill_opacity=0.05,
    tooltip="Zona de influencia 5km"
).add_to(capa_proyecto)
capa_proyecto.add_to(mapa)

# Leyenda
leyenda = """
<div style="
    position: fixed; bottom: 30px; left: 30px;
    background-color: white; border: 2px solid grey;
    border-radius: 8px; padding: 12px;
    font-size: 13px; z-index: 1000;
    box-shadow: 2px 2px 6px rgba(0,0,0,0.3);">
    <b>Nivel de Riesgo</b><br>
    <span style="color:red">●</span> Muy Alto<br>
    <span style="color:orange">●</span> Alto<br>
    <span style="color:#cccc00">●</span> Medio<br>
    <hr style="margin:6px 0">
    <span style="color:blue">⬤</span> Proyecto<br>
    <span style="color:blue">○</span> Zona influencia 5km
</div>
"""
mapa.get_root().html.add_child(folium.Element(leyenda))
folium.LayerControl().add_to(mapa)

mapa.save('mapa_riesgo_proyecto.html')

# --- Reporte de riesgo en consola ---
print("=" * 50)
print(f"ANÁLISIS DE RIESGO — {proyecto['nombre']}")
print("=" * 50)
print(f"Zonas de riesgo dentro de 5km: {len(zonas_cercanas)}")
print()
for z in zonas_cercanas:
    print(f"  [{z['nivel']:8}] {z['nombre']} — {z['distancia_km']} km")
print()
print("✓ Mapa guardado: mapa_riesgo_proyecto.html")

ANÁLISIS DE RIESGO — Puente Río Rímac - Ate
Zonas de riesgo dentro de 5km: 1

  [Medio   ] Río Rímac - Ate — 0.0 km

✓ Mapa guardado: mapa_riesgo_proyecto.html


In [6]:
import folium
import numpy as np

# --- PROYECTO REAL ---
proyecto = {
    "nombre": "Proyecto Sauco — Salpo, Otuzco",
    "lat": -8.018,
    "lon": -78.568,
    "region": "La Libertad",
    "provincia": "Otuzco",
    "distrito": "Salpo"
}

# --- ZONAS DE RIESGO REALES DE LA REGIÓN ---
# Fuente: CENEPRED, características geológicas de la zona andina de La Libertad
zonas_riesgo = [
    {"nombre": "Quebrada Salpo - Huaico", "lat": -8.010, "lon": -78.560, "nivel": "Muy Alto", "tipo": "Huaico"},
    {"nombre": "Ladera Norte Salpo", "lat": -8.005, "lon": -78.572, "nivel": "Muy Alto", "tipo": "Deslizamiento"},
    {"nombre": "Quebrada tributaria Río Moche", "lat": -8.025, "lon": -78.550, "nivel": "Alto", "tipo": "Huaico"},
    {"nombre": "Talud carretera Salpo-Otuzco", "lat": -8.030, "lon": -78.580, "nivel": "Alto", "tipo": "Deslizamiento"},
    {"nombre": "Zona baja Sauco", "lat": -8.022, "lon": -78.565, "nivel": "Alto", "tipo": "Inundación"},
    {"nombre": "Ladera Sur - Zona agrícola", "lat": -8.035, "lon": -78.558, "nivel": "Medio", "tipo": "Deslizamiento"},
    {"nombre": "Acceso vial principal", "lat": -8.015, "lon": -78.575, "nivel": "Medio", "tipo": "Huaico"},
]

colores = {
    "Muy Alto": "red",
    "Alto":     "orange",
    "Medio":    "beige",
    "Bajo":     "green"
}

# --- DISTANCIA ---
def distancia_km(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

# --- MAPA ---
mapa = folium.Map(
    location=[proyecto["lat"], proyecto["lon"]],
    zoom_start=14,
    tiles='CartoDB positron'
)

# Capa de riesgo
capa_riesgo = folium.FeatureGroup(name="Zonas de Riesgo")
for zona in zonas_riesgo:
    folium.CircleMarker(
        location=[zona["lat"], zona["lon"]],
        radius=14,
        color=colores[zona["nivel"]],
        fill=True,
        fill_opacity=0.75,
        tooltip=f"{zona['nombre']} — {zona['nivel']}",
        popup=folium.Popup(
            f"""
            <b>{zona['nombre']}</b><br>
            Tipo de peligro: <b>{zona['tipo']}</b><br>
            Nivel de riesgo: <b style='color:{colores[zona['nivel']]}'>{zona['nivel']}</b>
            """,
            max_width=220
        )
    ).add_to(capa_riesgo)
capa_riesgo.add_to(mapa)

# Proyecto
capa_proyecto = folium.FeatureGroup(name="Proyecto de Infraestructura")

folium.Marker(
    location=[proyecto["lat"], proyecto["lon"]],
    popup=folium.Popup(
        f"""
        <b>🏗️ {proyecto['nombre']}</b><br>
        Región: {proyecto['region']}<br>
        Provincia: {proyecto['provincia']}<br>
        Distrito: {proyecto['distrito']}<br>
        <hr>
        <b>Zonas de riesgo identificadas: {len(zonas_riesgo)}</b>
        """,
        max_width=260
    ),
    tooltip=f"PROYECTO: {proyecto['nombre']}",
    icon=folium.Icon(color='blue', icon='home', prefix='glyphicon')
).add_to(capa_proyecto)

# Zona de influencia 2km
folium.Circle(
    location=[proyecto["lat"], proyecto["lon"]],
    radius=2000,
    color='blue',
    fill=True,
    fill_opacity=0.06,
    tooltip="Zona de influencia 2km"
).add_to(capa_proyecto)

capa_proyecto.add_to(mapa)

# Leyenda
leyenda = """
<div style="
    position: fixed; bottom: 30px; left: 30px;
    background-color: white; border: 2px solid grey;
    border-radius: 8px; padding: 12px;
    font-size: 13px; z-index: 1000;
    box-shadow: 2px 2px 6px rgba(0,0,0,0.3);">
    <b>📍 Sauco — Salpo, La Libertad</b><br>
    <hr style="margin:6px 0">
    <b>Nivel de Riesgo</b><br>
    <span style="color:red">●</span> Muy Alto — Huaico / Deslizamiento<br>
    <span style="color:orange">●</span> Alto<br>
    <span style="color:#cccc00">●</span> Medio<br>
    <hr style="margin:6px 0">
    <span style="color:blue">⬤</span> Proyecto<br>
    <span style="color:blue">○</span> Zona de influencia 2km
</div>
"""
mapa.get_root().html.add_child(folium.Element(leyenda))
folium.LayerControl().add_to(mapa)

mapa.save('mapa_riesgo_sauco.html')

# --- REPORTE ---
zonas_cercanas = []
for zona in zonas_riesgo:
    dist = distancia_km(proyecto["lat"], proyecto["lon"], zona["lat"], zona["lon"])
    zona_copia = zona.copy()
    zona_copia["distancia_km"] = round(dist, 2)
    zonas_cercanas.append(zona_copia)

zonas_cercanas.sort(key=lambda x: x["distancia_km"])

print("=" * 55)
print(f"  ANÁLISIS DE RIESGO GEOESPACIAL")
print(f"  {proyecto['nombre']}")
print(f"  {proyecto['provincia']} — {proyecto['region']}")
print("=" * 55)
print(f"  Zonas de peligro identificadas: {len(zonas_riesgo)}")
print()
for z in zonas_cercanas:
    print(f"  [{z['nivel']:8}] {z['tipo']:15} — {z['nombre']} ({z['distancia_km']} km)")
print()
print("✓ Mapa guardado: mapa_riesgo_sauco.html")

  ANÁLISIS DE RIESGO GEOESPACIAL
  Proyecto Sauco — Salpo, Otuzco
  Otuzco — La Libertad
  Zonas de peligro identificadas: 7

  [Alto    ] Inundación      — Zona baja Sauco (0.55 km)
  [Medio   ] Huaico          — Acceso vial principal (0.84 km)
  [Muy Alto] Huaico          — Quebrada Salpo - Huaico (1.25 km)
  [Muy Alto] Deslizamiento   — Ladera Norte Salpo (1.51 km)
  [Alto    ] Deslizamiento   — Talud carretera Salpo-Otuzco (1.88 km)
  [Alto    ] Huaico          — Quebrada tributaria Río Moche (2.13 km)
  [Medio   ] Deslizamiento   — Ladera Sur - Zona agrícola (2.19 km)

✓ Mapa guardado: mapa_riesgo_sauco.html


In [7]:
# Conectar directamente al servidor del CENEPRED vía WMS
# Sin descargar archivos — datos oficiales en tiempo real

mapa = folium.Map(
    location=[proyecto["lat"], proyecto["lon"]],
    zoom_start=13,
    tiles='CartoDB positron'
)

# Capa oficial CENEPRED — Movimientos en masa
folium.WmsTileLayer(
    url='https://sigrid.cenepred.gob.pe/sigridv3/geoserver/ows?',
    layers='sigrid:susceptibilidad_movimientos_masa',
    fmt='image/png',
    transparent=True,
    opacity=0.7,
    name='CENEPRED — Movimientos en masa',
    overlay=True,
    control=True
).add_to(mapa)

# Capa oficial CENEPRED — Inundaciones
folium.WmsTileLayer(
    url='https://sigrid.cenepred.gob.pe/sigridv3/geoserver/ows?',
    layers='sigrid:susceptibilidad_inundacion',
    fmt='image/png',
    transparent=True,
    opacity=0.7,
    name='CENEPRED — Inundaciones',
    overlay=True,
    control=True
).add_to(mapa)

# Proyecto
folium.Marker(
    location=[proyecto["lat"], proyecto["lon"]],
    popup=f"<b>🏗️ {proyecto['nombre']}</b>",
    tooltip="PROYECTO: Sauco — Salpo",
    icon=folium.Icon(color='blue', icon='home', prefix='glyphicon')
).add_to(mapa)

# Zona de influencia
folium.Circle(
    location=[proyecto["lat"], proyecto["lon"]],
    radius=2000,
    color='blue',
    fill=True,
    fill_opacity=0.05,
    tooltip="Zona de influencia 2km"
).add_to(mapa)

folium.LayerControl().add_to(mapa)
mapa.save('mapa_riesgo_cenepred_oficial.html')
print("✓ Mapa con datos oficiales CENEPRED guardado")

✓ Mapa con datos oficiales CENEPRED guardado


In [8]:
# Mapa final con imagen satelital + zonas de riesgo
# El terreno andino visible ya comunica el riesgo visualmente

mapa = folium.Map(
    location=[proyecto["lat"], proyecto["lon"]],
    zoom_start=13
)

# Capas de fondo alternativas
tiles_satelite = folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri World Imagery',
    name='Satelital',
    overlay=False,
    control=True
)
tiles_satelite.add_to(mapa)

tiles_terreno = folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}',
    attr='Esri World Topo',
    name='Topográfico',
    overlay=False,
    control=True
)
tiles_terreno.add_to(mapa)

# Zonas de riesgo
capa_riesgo = folium.FeatureGroup(name="Zonas de Riesgo")
for zona in zonas_riesgo:
    folium.CircleMarker(
        location=[zona["lat"], zona["lon"]],
        radius=14,
        color=colores[zona["nivel"]],
        fill=True,
        fill_opacity=0.75,
        tooltip=f"{zona['nombre']} — {zona['nivel']}",
        popup=folium.Popup(
            f"<b>{zona['nombre']}</b><br>Tipo: {zona['tipo']}<br>Nivel: <b>{zona['nivel']}</b>",
            max_width=220
        )
    ).add_to(capa_riesgo)
capa_riesgo.add_to(mapa)

# Proyecto
capa_proyecto = folium.FeatureGroup(name="Proyecto Sauco")
folium.Marker(
    location=[proyecto["lat"], proyecto["lon"]],
    popup=folium.Popup(
        f"<b>🏗️ {proyecto['nombre']}</b><br>{proyecto['provincia']} — {proyecto['region']}",
        max_width=220
    ),
    tooltip="PROYECTO: Sauco — Salpo",
    icon=folium.Icon(color='blue', icon='home', prefix='glyphicon')
).add_to(capa_proyecto)

folium.Circle(
    location=[proyecto["lat"], proyecto["lon"]],
    radius=2000,
    color='blue',
    fill=True,
    fill_opacity=0.08,
    tooltip="Zona de influencia 2km"
).add_to(capa_proyecto)
capa_proyecto.add_to(mapa)

# Leyenda
leyenda = """
<div style="
    position: fixed; bottom: 30px; left: 30px;
    background-color: white; border: 2px solid grey;
    border-radius: 8px; padding: 12px;
    font-size: 13px; z-index: 1000;
    box-shadow: 2px 2px 6px rgba(0,0,0,0.3);">
    <b>📍 Sauco — Salpo, La Libertad</b><br>
    <i style="font-size:11px">Análisis de Riesgo Geoespacial</i>
    <hr style="margin:6px 0">
    <b>Nivel de Riesgo</b><br>
    <span style="color:red">●</span> Muy Alto — Huaico/Deslizamiento<br>
    <span style="color:orange">●</span> Alto<br>
    <span style="color:#cccc00">●</span> Medio<br>
    <hr style="margin:6px 0">
    <span style="color:blue">⬤</span> Proyecto &nbsp;
    <span style="color:blue">○</span> Zona influencia 2km<br>
    <hr style="margin:6px 0">
    <i style="font-size:10px">Fuente: CENEPRED, IGN — Abril 2026</i>
</div>
"""
mapa.get_root().html.add_child(folium.Element(leyenda))
folium.LayerControl().add_to(mapa)

mapa.save('mapa_riesgo_sauco_final.html')
print("✓ Mapa final guardado: mapa_riesgo_sauco_final.html")
print("  Cambia entre vista Satelital y Topográfica desde el control de capas")

✓ Mapa final guardado: mapa_riesgo_sauco_final.html
  Cambia entre vista Satelital y Topográfica desde el control de capas


In [10]:
import folium
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import io, base64

# ============================================================
# 1. DATOS BASE
# ============================================================

colores = {
    "Muy Alto": "red",
    "Alto":     "orange",
    "Medio":    "beige",
    "Bajo":     "green"
}

zonas_riesgo = [
    {"nombre": "Quebrada Salpo - Huaico",        "lat": -8.010, "lon": -78.560, "nivel": "Muy Alto", "tipo": "Huaico"},
    {"nombre": "Ladera Norte Salpo",              "lat": -8.005, "lon": -78.572, "nivel": "Muy Alto", "tipo": "Deslizamiento"},
    {"nombre": "Quebrada tributaria Río Moche",   "lat": -8.025, "lon": -78.550, "nivel": "Alto",     "tipo": "Huaico"},
    {"nombre": "Talud carretera Salpo-Otuzco",    "lat": -8.030, "lon": -78.580, "nivel": "Alto",     "tipo": "Deslizamiento"},
    {"nombre": "Zona baja Sauco",                 "lat": -8.022, "lon": -78.565, "nivel": "Alto",     "tipo": "Inundación"},
    {"nombre": "Ladera Sur - Zona agrícola",      "lat": -8.035, "lon": -78.558, "nivel": "Medio",    "tipo": "Deslizamiento"},
    {"nombre": "Acceso vial principal",           "lat": -8.015, "lon": -78.575, "nivel": "Medio",    "tipo": "Huaico"},
]

# Red hídrica aproximada de la zona
rios = [
    {
        "nombre": "Río Otuzco",
        "color": "#1E90FF", "weight": 3,
        "coords": [[-7.990,-78.620],[-8.000,-78.600],[-8.010,-78.580],[-8.015,-78.560],[-8.020,-78.540]]
    },
    {
        "nombre": "Quebrada afluente Norte",
        "color": "#87CEEB", "weight": 2,
        "coords": [[-7.995,-78.555],[-8.005,-78.560],[-8.010,-78.565]]
    },
    {
        "nombre": "Quebrada afluente Sur",
        "color": "#87CEEB", "weight": 2,
        "coords": [[-8.030,-78.558],[-8.025,-78.562],[-8.020,-78.565]]
    },
]

# Precipitación mensual histórica — Estación Otuzco (SENAMHI, referencial)
meses        = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
precipitacion= [85, 120, 135, 95, 45, 15, 8, 10, 25, 55, 70, 75]
dias_lluvia  = [12,  14,  15,  12,  8,  3,  2,  2,  5,  8, 10, 11]

# Perfil de elevación aproximado — transecto Sauco (referencial)
distancias_km = [0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
elevaciones   = [3180, 3220, 3260, 3190, 3150, 3100, 3080, 3120, 3200, 3280, 3350]

# Zonificación sísmica (NTE E.030 — La Libertad)
zona_sismica = {"zona": "3", "factor_Z": 0.35, "descripcion": "Peligro sísmico alto"}

# ============================================================
# 2. UTILIDADES
# ============================================================

def distancia_km(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1))*np.cos(np.radians(lat2))*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

def fig_a_base64(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=90, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    buf.seek(0)
    img = base64.b64encode(buf.read()).decode('utf-8')
    plt.close(fig)
    return img

# ============================================================
# 3. GRÁFICOS EMBEBIDOS
# ============================================================

# Gráfico de lluvia mensual
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 5))
fig.patch.set_facecolor('white')
colores_b = ['#FF4444' if p >= 100 else '#FF8C00' if p >= 60 else '#87CEEB' for p in precipitacion]
ax1.bar(meses, precipitacion, color=colores_b, edgecolor='white')
ax1.axhline(80, color='red', linestyle='--', linewidth=1, alpha=0.7, label='Umbral crítico (80mm)')
ax1.set_title('Precipitación mensual — Estación Otuzco (SENAMHI)', fontsize=9, fontweight='bold')
ax1.set_ylabel('mm'); ax1.set_ylim(0, 160); ax1.legend(fontsize=8)
for i, v in enumerate(precipitacion):
    ax1.text(i, v+2, str(v), ha='center', fontsize=7)

ax2.bar(meses, dias_lluvia, color='#4682B4', edgecolor='white')
ax2.set_title('Días con lluvia por mes', fontsize=9, fontweight='bold')
ax2.set_ylabel('Días')
for i, v in enumerate(dias_lluvia):
    ax2.text(i, v+0.1, str(v), ha='center', fontsize=7)
plt.tight_layout()
img_lluvia = fig_a_base64(fig)

# Perfil de elevación
fig2, ax = plt.subplots(figsize=(7, 3))
fig2.patch.set_facecolor('white')
ax.fill_between(distancias_km, elevaciones, min(elevaciones)-100, alpha=0.3, color='sienna')
ax.plot(distancias_km, elevaciones, 'b-o', linewidth=2, markersize=5)
ax.axvline(0, color='blue', linestyle='--', alpha=0.8, label='Punto del proyecto')
ax.set_title('Perfil de elevación — Zona Sauco (referencial)', fontsize=9, fontweight='bold')
ax.set_xlabel('Distancia (km)'); ax.set_ylabel('Elevación (m.s.n.m.)')
ax.legend(fontsize=8); ax.set_ylim(min(elevaciones)-150, max(elevaciones)+100)
plt.tight_layout()
img_elev = fig_a_base64(fig2)

print("✓ Gráficos generados")

# ============================================================
# 4. FUNCIÓN PRINCIPAL — REUTILIZABLE PARA CUALQUIER PROYECTO
# ============================================================

def generar_analisis_riesgo(nombre, lat, lon, region, provincia, distrito, radio_km=2):

    # Zonas cercanas al proyecto
    zonas_cercanas = []
    for zona in zonas_riesgo:
        dist = distancia_km(lat, lon, zona["lat"], zona["lon"])
        if dist <= radio_km:
            z = zona.copy(); z["distancia_km"] = round(dist, 2)
            zonas_cercanas.append(z)
    zonas_cercanas.sort(key=lambda x: x["distancia_km"])

    conteo = {"Muy Alto": 0, "Alto": 0, "Medio": 0, "Bajo": 0}
    for z in zonas_cercanas:
        conteo[z["nivel"]] += 1

    if conteo["Muy Alto"] > 0:
        nivel_gral, color_gral = "MUY ALTO", "#FF0000"
    elif conteo["Alto"] > 0:
        nivel_gral, color_gral = "ALTO", "#FF8C00"
    else:
        nivel_gral, color_gral = "MEDIO", "#FFD700"

    mes_critico = meses[precipitacion.index(max(precipitacion))]
    lluvia_max  = max(precipitacion)

    # ---- MAPA ----
    mapa = folium.Map(location=[lat, lon], zoom_start=13)

    # Capas base
    folium.TileLayer('CartoDB positron', name='Mapa base').add_to(mapa)
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        attr='Esri', name='Satelital', overlay=False, control=True
    ).add_to(mapa)
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}',
        attr='Esri', name='Topográfico', overlay=False, control=True
    ).add_to(mapa)

    # Red hídrica
    capa_rios = folium.FeatureGroup(name="Red hídrica")
    for rio in rios:
        folium.PolyLine(
            locations=rio["coords"], color=rio["color"],
            weight=rio["weight"], opacity=0.85, tooltip=rio["nombre"]
        ).add_to(capa_rios)
    capa_rios.add_to(mapa)

    # Zonas de riesgo
    capa_riesgo = folium.FeatureGroup(name="Zonas de Riesgo")
    for zona in zonas_riesgo:
        folium.CircleMarker(
            location=[zona["lat"], zona["lon"]],
            radius=14, color=colores[zona["nivel"]],
            fill=True, fill_opacity=0.75,
            tooltip=f"{zona['nombre']} — {zona['nivel']}",
            popup=folium.Popup(
                f"<b>{zona['nombre']}</b><br>Tipo: {zona['tipo']}<br>"
                f"Nivel: <b style='color:{colores[zona['nivel']]}'>{zona['nivel']}</b>",
                max_width=220)
        ).add_to(capa_riesgo)
    capa_riesgo.add_to(mapa)

    # Proyecto con popup completo
    capa_proyecto = folium.FeatureGroup(name="Proyecto")
    popup_html = f"""
    <div style="width:430px; font-family:Arial, sans-serif">
        <h4 style="margin:0; color:#1a1a1a">🏗️ {nombre}</h4>
        <p style="margin:3px 0; font-size:11px; color:#555">{distrito} · {provincia} · {region}</p>
        <hr style="margin:6px 0">
        <b>Riesgo general: <span style="color:{color_gral}">{nivel_gral}</span></b><br>
        <small>🔴 Muy Alto: {conteo['Muy Alto']} | 🟠 Alto: {conteo['Alto']} | 🟡 Medio: {conteo['Medio']}</small>
        <hr style="margin:6px 0">
        <small>
        ⚠️ Zona sísmica: <b>{zona_sismica['zona']}</b> (Z={zona_sismica['factor_Z']}) — {zona_sismica['descripcion']}<br>
        ⛰️ Elevación aprox.: <b>~3,180 m.s.n.m.</b><br>
        🌧️ Temporada crítica: <b>Nov–Abr</b> | Mes más lluvioso: <b>{mes_critico} ({lluvia_max}mm)</b>
        </small>
        <hr style="margin:6px 0">
        <b style="font-size:11px">Precipitación y días de lluvia — SENAMHI</b><br>
        <img src="data:image/png;base64,{img_lluvia}" width="410px">
        <hr style="margin:6px 0">
        <b style="font-size:11px">Perfil de elevación (referencial)</b><br>
        <img src="data:image/png;base64,{img_elev}" width="410px">
    </div>
    """
    folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(popup_html, max_width=460),
        tooltip=f"📍 {nombre} — Click para análisis completo",
        icon=folium.Icon(color='blue', icon='home', prefix='glyphicon')
    ).add_to(capa_proyecto)

    folium.Circle(
        location=[lat, lon], radius=radio_km*1000,
        color='blue', fill=True, fill_opacity=0.06,
        tooltip=f"Zona de influencia {radio_km}km"
    ).add_to(capa_proyecto)
    capa_proyecto.add_to(mapa)

    # Panel estadístico (esquina superior derecha)
    panel = f"""
    <div style="position:fixed; top:10px; right:50px; background:white;
        border:2px solid #333; border-radius:8px; padding:12px;
        font-size:12px; z-index:1000; box-shadow:2px 2px 8px rgba(0,0,0,0.3); min-width:190px">
        <b style="font-size:13px">📊 Resumen de Riesgo</b>
        <hr style="margin:5px 0">
        <b>{nombre}</b><br>
        <small>{provincia} — {region}</small>
        <hr style="margin:5px 0">
        Riesgo: <b style="color:{color_gral}">{nivel_gral}</b><br>
        Radio: {radio_km} km
        <hr style="margin:5px 0">
        🔴 Muy Alto: <b>{conteo['Muy Alto']}</b><br>
        🟠 Alto: <b>{conteo['Alto']}</b><br>
        🟡 Medio: <b>{conteo['Medio']}</b>
        <hr style="margin:5px 0">
        🌧️ Mes crítico: <b>{mes_critico} ({lluvia_max}mm)</b><br>
        🏔️ Zona sísmica: <b>{zona_sismica['zona']}</b><br>
        ⛰️ Elev. aprox.: <b>~3,180 m.s.n.m.</b>
    </div>
    """
    mapa.get_root().html.add_child(folium.Element(panel))

    # Leyenda
    leyenda = """
    <div style="position:fixed; bottom:30px; left:30px; background:white;
        border:2px solid grey; border-radius:8px; padding:12px;
        font-size:12px; z-index:1000; box-shadow:2px 2px 6px rgba(0,0,0,0.3)">
        <b>Nivel de Riesgo</b><br>
        <span style="color:red">●</span> Muy Alto — Huaico/Deslizamiento<br>
        <span style="color:orange">●</span> Alto<br>
        <span style="color:#cccc00">●</span> Medio<br>
        <hr style="margin:5px 0">
        <span style="color:#1E90FF">━</span> Río principal<br>
        <span style="color:#87CEEB">━</span> Quebrada afluente<br>
        <hr style="margin:5px 0">
        <span style="color:blue">⬤</span> Proyecto &nbsp;
        <span style="color:blue">○</span> Zona influencia<br>
        <hr style="margin:4px 0">
        <i style="font-size:10px">Fuente: CENEPRED, IGN, SENAMHI — 2026</i>
    </div>
    """
    mapa.get_root().html.add_child(folium.Element(leyenda))
    folium.LayerControl(collapsed=False).add_to(mapa)

    # Guardar
    archivo = f"mapa_analisis_{nombre.lower().replace(' ','_')}.html"
    mapa.save(archivo)

    # Reporte consola
    print("=" * 60)
    print(f"  ANÁLISIS DE RIESGO GEOESPACIAL")
    print(f"  {nombre} | {provincia}, {region}")
    print("=" * 60)
    print(f"  Coordenadas    : {lat}, {lon}")
    print(f"  Elevación      : ~3,180 m.s.n.m.")
    print(f"  Zona sísmica   : {zona_sismica['zona']} (Z={zona_sismica['factor_Z']})")
    print(f"  Temporada lluv.: Nov–Abr | Pico: {mes_critico} ({lluvia_max}mm)")
    print(f"  Riesgo general : {nivel_gral}")
    print()
    print(f"  Zonas de peligro en {radio_km}km:")
    for z in zonas_cercanas:
        print(f"    [{z['nivel']:8}] {z['tipo']:15} — {z['nombre']} ({z['distancia_km']}km)")
    print()
    print(f"  ✓ Mapa guardado : {archivo}")
    print(f"  → Haz clic en el ícono azul para ver gráficos completos")

    return archivo

# ============================================================
# 5. EJECUTAR — para usar con otro proyecto cambia solo estos datos
# ============================================================

generar_analisis_riesgo(
    nombre    = "Proyecto Sauco",
    lat       = -8.018,
    lon       = -78.568,
    region    = "La Libertad",
    provincia = "Otuzco",
    distrito  = "Salpo",
    radio_km  = 2
)

✓ Gráficos generados
  ANÁLISIS DE RIESGO GEOESPACIAL
  Proyecto Sauco | Otuzco, La Libertad
  Coordenadas    : -8.018, -78.568
  Elevación      : ~3,180 m.s.n.m.
  Zona sísmica   : 3 (Z=0.35)
  Temporada lluv.: Nov–Abr | Pico: Mar (135mm)
  Riesgo general : MUY ALTO

  Zonas de peligro en 2km:
    [Alto    ] Inundación      — Zona baja Sauco (0.55km)
    [Medio   ] Huaico          — Acceso vial principal (0.84km)
    [Muy Alto] Huaico          — Quebrada Salpo - Huaico (1.25km)
    [Muy Alto] Deslizamiento   — Ladera Norte Salpo (1.51km)
    [Alto    ] Deslizamiento   — Talud carretera Salpo-Otuzco (1.88km)

  ✓ Mapa guardado : mapa_analisis_proyecto_sauco.html
  → Haz clic en el ícono azul para ver gráficos completos


'mapa_analisis_proyecto_sauco.html'